# SHIPP Stage 8C — Silver listing content

`silver_listings` + primary image (`silver_listing_files`) + latest vision result (`vision_responses`) → `shipp_silver.silver_listing_content`

**Grain:** one row per current `listing_id` (every listing, whatever its status).
**`vision_status`:** `OK` · `FAILED` · `PENDING` (image not processed yet) · `NO_IMAGE` · `FILE_MISSING`.
Image fields are filled **only** when `vision_status = OK`. Emails/phone numbers are redacted from all text (spec §9.1).
`searchable_text` = title + category + condition + description + image description + visible features.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_listing_content`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [0]:
from rag import content
from rag.text import pii_rows_filter
from rag.vision import BRONZE_VISION_SCHEMA

TARGET_TABLE = SILVER_LISTING_CONTENT
assert table_exists(SILVER_LISTING_FILES), "FAILED: run rag/50_silver_listing_files first."

listings = spark.table(SILVER_LISTINGS)
primary = content.primary_image_files(spark.table(SILVER_LISTING_FILES))

if table_exists(BRONZE_VISION_RESPONSES):
    vision_rows = [r.asDict() for r in spark.table(BRONZE_VISION_RESPONSES).select(
        "call_id", "called_at", "listing_file_id", "file_sha256", "vision_endpoint", "prompt_version",
        "http_status", "response_body", "error_message").collect()]
else:
    vision_rows = []
vision_results = content.latest_vision_results(vision_rows, VISION_ENDPOINT, VISION_PROMPT_VERSION)
vision_df = spark.createDataFrame(vision_results, content.VISION_RESULT_SCHEMA)
print(f"Bronze vision rows: {len(vision_rows)} | results for {VISION_ENDPOINT} / {VISION_PROMPT_VERSION}: {len(vision_results)}")

## Step 1 — Build content

In [0]:
content_df = content.build_listing_content(listings, primary, vision_df, VISION_ENDPOINT, VISION_PROMPT_VERSION)
row_count = content_df.count()
display(content_df.groupBy("vision_status").count().orderBy("vision_status"))
display(content_df.select("listing_id", "vision_status", "image_description", "detected_labels",
                          "searchable_text", "vision_error").orderBy("listing_id"))

## Step 2 — Quality gate (the never-invented and privacy checks live here)

In [0]:
run_quality_gate({
    "duplicate listing_id": duplicate_key_count(content_df, "listing_id"),
    "null listing_id": null_count(content_df, "listing_id"),
    "null searchable_text": null_count(content_df, "searchable_text"),
    "vision_status not in vocabulary": content_df.filter(~F.col("vision_status").isin(VISION_STATUSES)).count(),
    "image fields without vision OK": content_df.filter(
        (F.col("vision_status") != "OK")
        & (F.col("image_description").isNotNull() | (F.size("detected_labels") > 0))).count(),
    "vision OK without description": content_df.filter(
        (F.col("vision_status") == "OK") & F.col("image_description").isNull()).count(),
    "email/phone left in searchable_text": content_df.filter(pii_rows_filter(F.col("searchable_text"))).count(),
    "row count != silver_listings": abs(row_count - listings.count()),
}, row_count=row_count)

## Step 3 — Write + exact schema + keys match Silver listings

In [0]:
written = write_overwrite(content_df, TARGET_TABLE)
assert written == row_count, f"WRITE VALIDATION FAILED: wrote {written}, validated {row_count}"
assert_exact_schema(TARGET_TABLE, content.CONTENT_COLUMNS)

stored = {r.listing_id for r in spark.table(TARGET_TABLE).select("listing_id").collect()}
expected = {r.listing_id for r in listings.select("listing_id").collect()}
assert stored == expected, f"FAILED: listing keys differ. missing={expected - stored} extra={stored - expected}"
print("PASS — one content row per current listing.")

counts = {r["vision_status"]: r["count"] for r in spark.table(TARGET_TABLE).groupBy("vision_status").count().collect()}
summary = {"table": TARGET_TABLE, "rows": written, "vision_status": counts, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))